# Does ABS momentum improve the top 100 merchants?

A controlled comparison of **Minimal + own growth** against the same approach plus one published ABS industry momentum feature. The outcome is actual gross fee revenue over the following three complete months.

The experiment uses the full eligible merchant pool. Unmatched merchants receive identical forecasts in both methods. Model settings, category proxies and the decision rule are fixed before evaluation. See [the reproduction guide](ABS_COMPARISON.md) for input contracts, tests and assumptions.

## 1. Paths and execution

On a fresh checkout, the next cell runs ingestion through evaluation. If results already exist, it displays them by default. Set `RUN_PIPELINE = True` for a fresh run. `REUSE_AUDITED_PANEL` is an explicit speed option only when the original archives and metadata have not changed. A rerun is a reproduction, not a new unseen holdout.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "code/abs_momentum_comparison.py").exists())
sys.path.insert(0, str(ROOT / "code"))
from abs_momentum_comparison import run

SOURCE = ROOT.parent / "project 2 materials"
OUTPUT = ROOT / "tables/analysis/abs_momentum_comparison"
RUN_PIPELINE = not (OUTPUT / "decision.json").exists()
REUSE_AUDITED_PANEL = False
OFFLINE = False

if RUN_PIPELINE:
    result = run(SOURCE, OUTPUT, offline=OFFLINE, reuse_panel=REUSE_AUDITED_PANEL)
else:
    print("Displaying the completed frozen experiment. Set RUN_PIPELINE=True to reproduce it.")

Displaying the completed frozen experiment. Set RUN_PIPELINE=True to reproduce it.


## 2. Data quality and category coverage

Only March 2021-September 2022 are complete merchant months. Valid large transactions are retained. Missing merchant metadata is reported separately, and transaction-free months are zero-filled only after source-day coverage is verified.

The mapping is an analyst crosswalk, including explicit retail-led assumptions for mixed labels. All other categories remain unmatched.

In [2]:
audit = json.loads((OUTPUT / "data_audit.json").read_text(encoding="utf-8"))
display(pd.Series(audit, name="value").to_frame())
mapping = pd.read_parquet(OUTPUT / "category_mapping.parquet")
display(mapping)
coverage = pd.read_parquet(OUTPUT / "coverage.parquet")
display(coverage.loc[coverage.origin.eq("2022-06-30")])
eligibility = pd.read_parquet(OUTPUT / "eligibility.parquet")
display(eligibility.loc[eligibility.origin.eq(pd.Timestamp("2022-06-30"))])

,value
source_orders,14195505
unique_order_ids,14195505
source_sales,2359703946.358811
source_first_day,2021-02-28
source_last_day,2022-10-26
complete_days,606
complete_month_orders,13440766
complete_month_sales,2234541395.35951
excluded_partial_month_orders,754739
unknown_complete_month_orders,549947


,category,series_id,match_quality,mapping_assumption
0,"digital goods: books, movies, music",,unmapped,Digital publishing/downloads/streaming are not...
1,artist supply and craft shops,A3349395V,approximate,Retail-led art materials and craft supplies; b...
2,"computer programming, data processing, and int...",,unmapped,Professional technology services are not compu...
3,shoe shops,A3349393R,direct,Footwear retail is included; the ABS subgroup ...
4,"furniture, home furnishings and equipment shop...",A3349785A,conditional,Assume retail furniture/home furnishings domin...
5,"gift, card, novelty, and souvenir shops",A3349395V,approximate,Assume store-based gift/card/novelty retail; b...
6,"computers, computer peripheral equipment, and ...",A3349392L,conditional,Assume hardware-led retail rather than softwar...
7,"florists supplies, nursery stock, and flowers",,unmapped,Mixed flower retail/garden supplies/wholesale ...
8,tent and awning shops,,unmapped,Camping equipment and building-related awnings...
9,"cable, satellite, and other pay television and...",,unmapped,Subscription services are outside the selected...


,origin,abs_matched,merchants,recent_fees,merchant_share_pct,recent_fee_share_pct
16,2022-06-30,False,1828,6.922828e+06,47.308489,46.716805
17,2022-06-30,True,2036,7.895882e+06,52.691511,53.283195


,origin,eligibility_reason,size
24,2022-06-30,eligible,3864
25,2022-06-30,no_sales_in_previous_three_months,114
26,2022-06-30,no_sales_in_recent_three_months,48


## 3. Historical information and features

Recent sales are the latest three months divided by three. Own growth is latest-three-month sales / preceding-three-month sales minus one. The model uses log recent sales and log(1 + own growth).

ABS momentum uses the same three-versus-three ratio within one historical Table 12 release. Each training row uses the vintage published at its own origin. National series are identified by their validated IDs and the workbook geography **Total (State)**.

At June 30, May's Table 12 figures were not yet published: the final forecast uses **April 2022**. ABS momentum is nominal spending growth, not inflation-adjusted demand growth.

In [3]:
abs_features = pd.read_parquet(OUTPUT / "abs_features.parquet")
release_timing = abs_features[["origin", "abs_reference_month", "abs_published_at"]].drop_duplicates()
display(release_timing)
features = pd.read_parquet(OUTPUT / "merchant_features.parquet")
display(features.loc[features.origin.eq("2022-06-30"),
    ["merchant_abn", "category", "recent_monthly_sales", "own_growth", "abs_momentum", "eligible"]].head(10))

,origin,abs_reference_month,abs_published_at
0,2021-08-31,2021-06-01,2021-08-04 11:30:00+10:00
7,2021-09-30,2021-07-01,2021-09-03 11:30:00+10:00
14,2021-10-31,2021-08-01,2021-10-05 11:30:00+11:00
21,2021-11-30,2021-09-01,2021-11-04 11:30:00+11:00
28,2021-12-31,2021-10-01,2021-12-02 11:30:00+11:00
35,2022-01-31,2021-11-01,2022-01-11 11:30:00+11:00
42,2022-02-28,2021-12-01,2022-02-07 11:30:00+11:00
49,2022-03-31,2022-01-01,2022-03-04 11:30:00+11:00
56,2022-06-30,2022-04-01,2022-06-02 11:30:00+10:00


,merchant_abn,category,recent_monthly_sales,own_growth,abs_momentum,eligible
32208,10023283211,"furniture, home furnishings and equipment shop...",36468.849484,0.171662,-0.003917,True
32209,10142254217,"cable, satellite, and other pay television and...",6305.677604,0.299740,NaN,True
32210,10165489824,"jewelry, watch, clock, and silverware shops",5172.465002,0.418838,NaN,True
32211,10187291046,"watch, clock, and jewelry repair shops",2378.941063,0.315555,NaN,True
32212,10192359162,"music shops - musical instruments, pianos, and...",6929.258013,-0.144774,0.007269,True
32213,10206519221,"gift, card, novelty, and souvenir shops",16555.478815,0.173800,0.007269,True
32214,10255988167,"computers, computer peripheral equipment, and ...",14815.596803,0.118398,0.007636,True
32215,10264435225,"watch, clock, and jewelry repair shops",29343.987915,0.224755,NaN,True
32216,10279061213,"computer programming, data processing, and int...",7776.575829,0.066690,NaN,True
32217,10323485998,"furniture, home furnishings and equipment shop...",69862.071078,0.282477,-0.003917,True


## 4. Fixed models and chronology

Tweedie mean regression uses power 1.5, log link, alpha 0.1 and an intercept. Training-only scaling, equal row weights and the same internal predictors apply to both matched models. The unmatched baseline is shared. Take rate converts predicted sales to gross fees; it is not an additional sales predictor.

Each model predicts total next-three-month sales. Only training rows whose full target window has ended can enter a fit. The final June 30 fit uses August 2021-March 2022 feature origins; July-September outcomes are attached only after forecasts and selections are saved.

Both policies rank predicted fees and choose exactly 100, breaking ties by ABN. The repeat-sales benchmark simply forecasts the latest three months again. There is no risk penalty or diversification constraint in this return-feature experiment.

In [4]:
protocol = json.loads((OUTPUT / "protocol.json").read_text(encoding="utf-8"))
display(pd.Series(protocol["model"], name="frozen setting").to_frame())
models = json.loads((OUTPUT / "fitted_models.json").read_text(encoding="utf-8"))
fit_summary = []
for origin, components in models.items():
    for component, details in components.items():
        fit_summary.append({"origin": origin, "component": component,
                            "training_rows": details["training_rows"],
                            "distinct_training_origins": len(details["training_origins"]),
                            "iterations": details["iterations"]})
display(pd.DataFrame(fit_summary))

,frozen setting
power,1.5
alpha,0.1
link,log
fit_intercept,True
solver,lbfgs
max_iter,2000
tol,0.000001


,origin,component,training_rows,distinct_training_origins,iterations
0,2021-12-31,matched_baseline,4062,2,10
1,2021-12-31,matched_enhanced,4062,2,10
2,2021-12-31,unmatched_shared,3655,2,9
3,2022-01-31,matched_baseline,6106,3,10
4,2022-01-31,matched_enhanced,6106,3,10
5,2022-01-31,unmatched_shared,5482,3,9
6,2022-02-28,matched_baseline,8156,4,10
7,2022-02-28,matched_enhanced,8156,4,10
8,2022-02-28,unmatched_shared,7327,4,9
9,2022-03-31,matched_baseline,10209,5,10


## 5. Selection performance and forecast diagnostics

The primary comparison is actual gross fees from the two selections. The retention rule requires **at least 1% final uplift and positive uplift in at least three of four earlier windows**.

MAE, WAPE and signed forecast bias are secondary diagnostics. Earlier windows overlap and are not independent replications.

In [5]:
folds = pd.read_parquet(OUTPUT / "fold_metrics.parquet")
display(folds)
forecast_metrics = pd.read_parquet(OUTPUT / "forecast_metrics.parquet")
display(forecast_metrics.loc[forecast_metrics.origin.eq("2022-06-30")])

,origin,baseline_fees,enhanced_fees,repeat_sales_fees,delta_fees,uplift_pct,top100_overlap,replacements,largest_absolute_change
0,2021-12-31,4.115846e+06,4.115846e+06,4.123036e+06,0.000000,0.00000,100,0,0.00000
1,2022-01-31,4.409658e+06,4.409658e+06,4.417297e+06,0.000000,0.00000,100,0,0.00000
2,2022-02-28,4.963514e+06,4.961822e+06,4.960990e+06,-1691.579476,-0.03408,99,1,23900.24041
3,2022-03-31,5.150602e+06,5.150602e+06,5.147926e+06,0.000000,0.00000,100,0,0.00000
4,2022-06-30,5.740017e+06,5.740017e+06,5.742167e+06,0.000000,0.00000,100,0,0.00000


,origin,method,cohort,merchants,mae,wape_pct,bias_dollars,bias_pct
24,2022-06-30,baseline,full_pool,3864,872.592437,20.513485,-3.017275e+06,-18.357174
25,2022-06-30,baseline,matched,2036,879.709474,20.375754,-1.609252e+06,-18.307147
26,2022-06-30,enhanced,full_pool,3864,862.941710,20.286609,-2.975100e+06,-18.100578
27,2022-06-30,enhanced,matched,2036,861.393950,19.951531,-1.567076e+06,-17.827352
28,2022-06-30,repeat_sales,full_pool,3864,557.799995,13.113134,-1.617781e+06,-9.842617
29,2022-06-30,repeat_sales,matched,2036,573.089378,13.273846,-8.944107e+05,-10.174981


## 6. Which merchants changed?

Entrants and removals explain the full fee difference because merchants selected by both methods cancel out. Inspect their categories and proxy assumptions before interpreting a small aggregate gain. These contributions do not identify causal effects.

In [6]:
changes = pd.read_parquet(OUTPUT / "selection_changes.parquet")
final_changes = changes.loc[changes.origin.eq("2022-06-30")]
display(final_changes[["merchant_abn", "category", "match_quality", "change", "actual_fees", "fee_contribution"]]
        .sort_values("fee_contribution", key=abs, ascending=False))
composition = pd.read_parquet(OUTPUT / "category_composition.parquet")
display(composition.loc[composition.origin.eq("2022-06-30")])
selected = pd.read_parquet(OUTPUT / "final_selected_merchants.parquet")
display(selected.loc[selected.method.eq("enhanced"),
    ["rank", "merchant_abn", "name", "category", "predicted_fees", "actual_fees"]].head(10))

,merchant_abn,category,match_quality,change,actual_fees,fee_contribution


,origin,method,category,match_quality,merchants,actual_fees,predicted_fees
288,2022-06-30,baseline,"antique shops - sales, repairs, and restoratio...",conditional,3,154500.957734,117116.030473
289,2022-06-30,baseline,art dealers and galleries,conditional,2,109055.320591,88724.831456
290,2022-06-30,baseline,artist supply and craft shops,approximate,5,348639.093013,262981.925118
291,2022-06-30,baseline,bicycle shops - sales and service,conditional,2,85211.940112,68325.413606
292,2022-06-30,baseline,"books, periodicals, and newspapers",direct,6,230502.064381,182094.100738
...,...,...,...,...,...,...,...
355,2022-06-30,repeat_sales,shoe shops,direct,3,137007.382710,123600.250109
356,2022-06-30,repeat_sales,"stationery, office supplies and printing and w...",approximate,3,154465.381686,139261.013658
357,2022-06-30,repeat_sales,telecom,unmapped,2,144270.555223,124823.471278
358,2022-06-30,repeat_sales,tent and awning shops,unmapped,12,742971.708689,662142.652519


,rank,merchant_abn,name,category,predicted_fees,actual_fees
100,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shop...",86175.540912,110968.155950
101,2,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops",81042.893548,106845.154252
102,3,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses",80975.472482,107923.823178
103,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",79736.308102,105875.832244
104,5,38700038932,Etiam Bibendum Industries,tent and awning shops,79143.420644,102245.704952
105,6,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops",76869.557396,100445.208528
106,7,96680767841,Ornare Limited,motor vehicle supplies and new parts,76601.239144,99379.520099
107,8,63123845164,Odio Phasellus Institute,artist supply and craft shops,74279.111780,96817.492414
108,9,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and...",74255.085807,98100.039800
109,10,64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and...",72963.718797,95940.284212


## 7. Decision and limitations

The report applies the frozen rule without tuning. The available history is short and synthetic; broad ABS proxies may not describe merchant-specific demand. Gross fee revenue excludes credit losses, funding and operating costs. Merchant categories and percentage take rates are assumed static and known, and historical sales stand in for hypothetical selection outcomes.

In [7]:
display(Markdown((OUTPUT / "findings.md").read_text(encoding="utf-8")))

# ABS momentum controlled comparison

**Decision: retain Minimal + own growth; insufficient evidence to add ABS momentum.**

Final July-September 2022 uplift: **+0.0000% (A$+0.00)**. Positive earlier tests: **0/4**. Required: final uplift >=1% and at least 3/4 earlier tests positive.

| Selection date | Baseline fees | With ABS fees | Repeat-sales fees | Uplift | Shared selections |
|---|---:|---:|---:|---:|---:|
| 2021-12-31 | 4,115,845.79 | 4,115,845.79 | 4,123,036.44 | +0.0000% | 100/100 |
| 2022-01-31 | 4,409,657.85 | 4,409,657.85 | 4,417,296.84 | +0.0000% | 100/100 |
| 2022-02-28 | 4,963,513.56 | 4,961,821.98 | 4,960,990.46 | -0.0341% | 99/100 |
| 2022-03-31 | 5,150,601.96 | 5,150,601.96 | 5,147,926.14 | +0.0000% | 100/100 |
| 2022-06-30 | 5,740,017.19 | 5,740,017.19 | 5,742,167.11 | +0.0000% | 100/100 |

## Forecast accuracy and benchmark

Final full-pool WAPE: baseline 20.51%, with ABS 20.29%, repeat-sales benchmark 13.11% (lower is better). An improvement in average forecast accuracy need not change the top 100.

The repeat-sales selection earned A$+2,149.92 (+0.0375%) relative to the fitted baseline. Retaining the simpler feature set does not establish that this fixed regression is the best ranking model.

## What changed in the final selection

ABS replaced 0 merchants. Entrants and removals reconcile exactly to the fee difference.


## Data and interpretation

Audited 14,195,505 unique orders across 606 source days. Used 13,440,766 orders in complete months before metadata exclusions; 549,947 of those belong to merchants without metadata.

The mapping covers 13/25 categories (2,121/4,026 metadata merchants before activity eligibility). Coverage by origin and recent fees is in coverage.csv. Category matches include documented proxies.

At the final cutoff, 3,864 merchants were eligible, including 2,036 with ABS matches.

Gross fees are recorded sales multiplied by the assumed percentage take rate. They exclude defaults, funding costs and operating costs. Categories and rates are assumed stable and known throughout the study; historical recorded sales stand in for outcomes of hypothetical merchant selection.

The final June 30 selection used the April 2022 ABS release. Historical workbooks and release pages are cached with publication provenance. They are the currently accessible dated ABS release files; independent captures of the exact original publication bytes were not available.

Only 19 complete merchant months are available. Earlier evaluation windows overlap; the initial fit has two distinct training origins and the final fit eight. Thousands of merchants do not create thousands of independent macroeconomic observations. Results support a pilot decision, not statistical significance, causal impact, or stability across economic cycles. Synthetic merchant demand may not track real ABS spending.

The repeat-sales benchmark is a sanity check. Forecast MAE/WAPE/bias, category composition and mapping attribution are supporting diagnostics; they do not replace the frozen top-100 decision rule.

No model tuning, category changes, risk penalties or extra predictors were introduced after evaluation.
